In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import anthropic

client = anthropic.Anthropic()


message = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello, Claude"}],
)
print(message)

print(message.content[0].text)

Hello! Nice to meet you. How can I help you today? Whether you have a question, need help with a project, or just want to chat, I'm here.


In [ ]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [12]:
import requests

quests_url = "https://datatalksclub.github.io/faq/json/llm-zoomcamp.json"
docs_url = "https://datatalksclub.github.io/faq/json/courses.json"


response = requests.get(docs_url)
courses_raw = response.json()

In [18]:
documents = []
url_prefix = "https://datatalksclub.github.io/faq"

for course in courses_raw :
    url = f'{url_prefix}{course["path"]}'

    course_response = requests.get(url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1456

In [19]:
from minsearch import Index

index= Index(
    text_fields=['question','section','answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [22]:
question = 'I just discovered the course can I join it ?'

index.search(question,
             filter_dict={'course': 'llm-zoomcamp'}, num_results=5)

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [23]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question':2.0, 'section': 0.5}
    filter_dict= {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
        )

search_results = search(question)

In [24]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '5cc511f85b',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Does the course certificate show the number of course hours?',
  'answer': 'No. The certificate does not 

In [25]:
def build_context(search_results):
    lines = []

    for elem in search_results:
        lines.append(elem['section'])
        lines.append(f'Q :{elem["question"]}')
        lines.append(f'A :{elem["answer"]}')
        lines.append('')

    return '\n'.join(lines).strip()

## BUILD Prompt

In [26]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q :I just discovered the course. Can I still join?
A :Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q :Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A :You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q :Does the course certificate show the number of course hours?
A :No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q :Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A :Notebooks are great for learning and experimenting, but real AI workflows need more than a script that runs once: scheduling, retri

In [27]:

INSTRUCTIONS= '''
Your task is to answer questions from the course participants based on the provided context.

Use the context to find relevant information and provide accurate answers. if the answer is not found in the context, 
respond with "I don't know."
'''

USER_PROMPT_TEMPLATE = f'''
Question:
{question}

Context:
{context}
'''

In [ ]:
USER_PROMPT_TEMPLATE.format(question=question,context=context)

'\nQuestion:\nI just discovered the course can I join it ?\n\nContext:\nGeneral Course-Related Questions\nQ :I just discovered the course. Can I still join?\nA :Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ :Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA :You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ :Does the course certificate show the number of course hours?\nA :No. The certificate does not state a total number of hours.\n\nModule 3: Orchestration\nQ :Why do we need orchestration / Kestra — can\'t I just run the code in a notebook?\nA :Notebooks are great for learning and exper

In [31]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(question=question,context=context)

    return prompt.strip()


prompt = build_prompt(question, search_results)
print(prompt)

Question:
I just discovered the course can I join it ?

Context:
General Course-Related Questions
Q :I just discovered the course. Can I still join?
A :Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q :Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A :You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q :Does the course certificate show the number of course hours?
A :No. The certificate does not state a total number of hours.

Module 3: Orchestration
Q :Why do we need orchestration / Kestra — can't I just run the code in a notebook?
A :Notebooks are great for learning and experimenting, but real AI wo

In [36]:
import anthropic

client = anthropic.Anthropic()

message = client.messages.create(
    model="claude-opus-5-5",
    max_tokens=1024,
    system="You are a helpful assistant. Answer in one short sentence.",
    messages=[{"role": "user", "content": "Hello, Claude"}],
)

print(message.content[0].text)


Hello! How can I help you today?


In [40]:
import anthropic

client = anthropic.Anthropic()


def llm(instructions, user_prompt, model='claude-opus-5-5'):
    messages = [{"role": "user", "content": user_prompt}]
    
    response = client.messages.create(
        model=model,
        max_tokens=1024,
        system=instructions,
        messages=messages,
    )

    return next(b.text for b in response.content if b.type == "text")



In [41]:
def rag(query, model='claude-opus-5-5'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)

    return answer

In [42]:
answer = rag(question)
print(answer)

Yes, you can still join. If you want a certificate, though, you need to submit your project while submissions are still being accepted.

A few related points:

- **No registration confirmation needed:** You can start learning and submitting homework (while the form is open) without registering. Submissions aren't checked against a registered list. Registration only helps gauge interest before the start date.
- **Certificate requirements:** You need to finish a capstone project and complete the required peer reviews. Homework isn't required for the certificate. You can study and prepare your project at your own pace, but the project submission and peer reviews must happen while a live cohort is accepting them.
